# M3: z-Transformation

Ja, beide können die z-Transformierte **symbolisch** berechnen:

- **Matlab** (Symbolic Math Toolbox): `syms n w; ztrans(cos(w*n))`, Rücktransformation mit `iztrans`.
- **Python**: [lcapy](https://lcapy.readthedocs.io/en/latest/) (baut auf sympy auf): `lc.cos(w*n).ZT()`, Rücktransformation mit `.IZT()`.

Beide berechnen die **einseitige** z-Transformation $F(z) = \sum_{n=0}^{\infty} f(n)\, z^{-n}$.
Das Produkt mit der Sprungfolge $u(n)$ aus der Aufgabe steckt also schon in der Definition.

In [ ]:
import lcapy as lc
import numpy as np
import sympy as sp
from lcapy.discretetime import n
from matplotlib import pyplot as plt
from plotly import graph_objects as go
from plotly.subplots import make_subplots

w = lc.symbol("w")

## a) $f(n) = \cos(n\omega)\, u(n)$

In [ ]:
F_cos = lc.cos(w * n).ZT()
F_cos

## b) $f(n) = \sin(n\omega)\, u(n)$

In [ ]:
F_sin = lc.sin(w * n).ZT()
F_sin

Beide haben denselben Nenner $z^2 - 2z\cos\omega + 1 = (z - e^{j\omega})(z - e^{-j\omega})$:
zwei Pole **auf** dem Einheitskreis beim Winkel $\pm\omega$. Die Reihe konvergiert für $|z| > 1$.

## c) $f(n) = n^4\, u(n)$

In [ ]:
F_n4 = (n**4).ZT()
sp.factor(F_n4.sympy)

Ein fünffacher Pol bei $z = 1$, Konvergenz für $|z| > 1$.
Allgemein gilt $\mathcal{Z}\{n^k u(n)\} = \dfrac{z\, A_k(z)}{(z-1)^{k+1}}$; die Zählerkoeffizienten $1, 11, 11, 1$ sind die Euler-Zahlen.
Man erhält das Ergebnis auch per Hand: $k$-mal die Regel $\mathcal{Z}\{n f(n)\} = -z \frac{d}{dz} F(z)$ auf $\frac{z}{z-1}$ anwenden.

In [ ]:
z = sp.Symbol("z")
F = z / (z - 1)
for _ in range(4):
    F = sp.factor(-z * sp.diff(F, z))
F

## Probe über die Definition

Für ein $z$ im Konvergenzgebiet (hier $z = 1.5$) muss die Partialsumme $\sum_{n=0}^{N} f(n) z^{-n}$ gegen den geschlossenen Ausdruck gehen.
Für $a^n u(n)$ liefert `sp.summation` sogar die Konvergenzbedingung mit.

In [ ]:
omega, z0, N = np.pi / 4, 1.5, 400
k = np.arange(N)
for name, f, F in [
    ("cos", np.cos(omega * k), F_cos),
    ("sin", np.sin(omega * k), F_sin),
    ("n^4", k.astype(float) ** 4, F_n4),
]:
    partial_sum = np.sum(f * z0 ** (-k.astype(float)))
    closed_form = complex(F.subs(w, omega).sympy.subs(lc.z.sympy, z0))
    print(f"{name}: Summe {partial_sum:.6f}, Formel {closed_form.real:.6f}")

a, m = sp.symbols("a m")
sp.summation(a**m * z ** (-m), (m, 0, sp.oo))

## Pole und Nullstellen

Alle Ergebnisse in einer Figure, einmal interaktiv mit Plotly und einmal mit matplotlib.
Oben die Folgen für $\omega = \pi/4$, unten die Pol-Nullstellen-Diagramme mit dem Einheitskreis.

In [ ]:
def poles_zeros(F, omega):
    expr = sp.together(F.subs(w, omega).sympy)
    variable = next(iter(expr.free_symbols))
    num, den = sp.fraction(expr)

    def roots(p):
        return [
            complex(r)
            for r, mult in sp.roots(sp.Poly(p, variable)).items()
            for _ in range(mult)
        ]

    return np.array(roots(den)), np.array(roots(num))


titles = ("cos(ωn) u(n)", "sin(ωn) u(n)", "n⁴ u(n)")
sequences = [np.cos(omega * k[:25]), np.sin(omega * k[:25]), k[:25].astype(float) ** 4]
transforms = [F_cos, F_sin, F_n4]
phi = np.linspace(0, 2 * np.pi, 400)

fig = make_subplots(
    rows=2, cols=3, subplot_titles=titles + ("", "", ""), vertical_spacing=0.12
)
for col, (x, F) in enumerate(zip(sequences, transforms, strict=True), start=1):
    fig.add_trace(
        go.Scatter(
            x=k[:25], y=x, mode="markers", marker_color="royalblue", showlegend=False
        ),
        row=1,
        col=col,
    )
    for kk, xx in zip(k[:25], x, strict=True):
        fig.add_shape(
            type="line",
            x0=kk,
            x1=kk,
            y0=0,
            y1=xx,
            line=dict(color="royalblue", width=1),
            row=1,
            col=col,
        )
    poles, zeros = poles_zeros(F, omega)
    fig.add_trace(
        go.Scatter(
            x=np.cos(phi),
            y=np.sin(phi),
            mode="lines",
            line=dict(color="gray", dash="dot"),
            showlegend=False,
        ),
        row=2,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=np.real(poles),
            y=np.imag(poles),
            mode="markers",
            marker=dict(symbol="x", size=11, color="crimson"),
            name="Pole",
            showlegend=col == 1,
        ),
        row=2,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=np.real(zeros),
            y=np.imag(zeros),
            mode="markers",
            marker=dict(symbol="circle-open", size=11, color="seagreen"),
            name="Nullstellen",
            showlegend=col == 1,
        ),
        row=2,
        col=col,
    )
    fig.update_xaxes(range=[-1.6, 1.6], title_text="Re z", row=2, col=col)
    fig.update_yaxes(range=[-1.6, 1.6], scaleanchor=f"x{col + 3}", row=2, col=col)
    fig.update_xaxes(title_text="n", row=1, col=col)
fig.update_layout(height=700, title="z-Transformierte für ω = π/4")
fig.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), layout="constrained")
for col, (x, F, title) in enumerate(zip(sequences, transforms, titles, strict=True)):
    axes[0, col].stem(k[:25], x, basefmt="k-")
    axes[0, col].set(title=title, xlabel="n")
    poles, zeros = poles_zeros(F, omega)
    ax = axes[1, col]
    ax.plot(np.cos(phi), np.sin(phi), ":", color="gray")
    ax.plot(poles.real, poles.imag, "x", color="crimson", markersize=10, label="Pole")
    ax.plot(
        zeros.real,
        zeros.imag,
        "o",
        mfc="none",
        color="seagreen",
        markersize=10,
        label="Nullstellen",
    )
    ax.set(
        xlim=(-1.6, 1.6), ylim=(-1.6, 1.6), aspect="equal", xlabel="Re z", ylabel="Im z"
    )
axes[1, 0].legend(loc="lower left")
axes[1, 2].annotate(
    "5-facher Pol", (1, 0), (0.1, 0.8), arrowprops=dict(arrowstyle="->")
)
plt.show()